In [ ]:

import numpy as np
from scipy.optimize import brentq
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})


In [ ]:
fileVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/Tests/RotatingMonopole/rotating_monopole_m005_lu_3d/microphonesCylinder/farfield1/microphone0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/Tests/RotatingMonopole/rotating_monopole_m005_lu_3d/microphonesCylinder/farfield1/microphone0001.txt",
]

cs = 1.0 / np.sqrt(3.0)
rho0 = 1.0
A = 1.0e-3
f0 = 0.02                       # oscillation frequency [1/step]
a = 35.0                        # rotation radius
Om = 2.0 * np.pi / 1000.0       # rotation angular velocity [rad/step]
w0 = 2.0 * np.pi * f0
M0 = 0.05                       # mean-flow Mach (+x)
obs = np.array([-100.0, 0.0, 0.0])

labelVec = [
    "Present LBM D3Q19 Regularized-BGK",
    "Present LBM D3Q19 Regularized-BGK",
    # "Present LBM D3Q19 Regularized-BGK sphere",
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]


U0 = M0 * cs
betaSq = 1.0 - M0 * M0

obs_used = obs.copy()

In [ ]:
#!/usr/bin/env python3
"""Exact solution of the 3D rotating oscillating monopole in uniform flow and
comparison with the FWH microphone output of rotating_monopole_m005_lu_3d.

Source (lattice units, from case.hpp):
    mass source   S(x,t) = A sin(w0 t) * delta(x - x_s(t)),  A = 1e-3
    x_s(t) = a*(cos(Om t), sin(Om t), 0),  a = 35,  Om = 2*pi/1000 rad/step
    w0 = 2*pi*0.02 rad/step  (carrier is the 20th rotation harmonic)
Medium: cs = 1/sqrt(3), rho0 = 1, uniform flow U0 = 0.05*cs along +x.

The naive "instantaneous position" formula (stationary-monopole solution with
x_s plugged in at observer time t) is NOT exact: the source moves at tip Mach
0.38, so the retarded time and the Doppler factor matter (|M_r| up to ~0.5).

Exact solution (subsonic, single retarded root):
    solve tau from   t = tau + sigma(tau)/cs
        beta^2 = 1 - M0^2,   r = x_obs - x_s(tau)
        R*     = sqrt(r_x^2 + beta^2 (r_y^2 + r_z^2))        (stretched radius)
        sigma  = (R* - M0 r_x)/beta^2                        (emission distance)
        g(tau) = tau + sigma(tau)/cs,   Doppler = g'(tau)
    potential   phi = -A sin(w0 tau) / (4 pi R* g')
    pressure    p'  = -rho0 (d/dt + U0 d/dx) phi
"""

def src_pos(t):
    return np.array([a * np.cos(Om * t), a * np.sin(Om * t), 0.0])


def _geo(tau):
    r = obs_used - src_pos(tau)
    Rstar = np.sqrt(r[0] ** 2 + betaSq * (r[1] ** 2 + r[2] ** 2))
    sigma = (Rstar - M0 * r[0]) / betaSq
    return Rstar, sigma


def _g(tau):
    return tau + _geo(tau)[1] / cs


def phi(t):
    """Retarded-time potential at (obs_used, t)."""
    global obs_used
    tau = brentq(lambda s: _g(s) - t, t - 600.0, t - 1e-12, xtol=1e-13)
    h = 1e-6
    gp = (_g(tau + h) - _g(tau - h)) / (2 * h)      # Doppler factor g'(tau)
    Rstar, _ = _geo(tau)
    return -A * np.sin(w0 * tau) / (4 * np.pi * Rstar * gp)


def p_prime(t, dt=1e-3, dx=1e-3):
    """p' = -rho0 (d/dt + U0 d/dx) phi, by finite differences on the exact phi."""
    global obs_used
    pt = (phi(t + dt) - phi(t - dt)) / (2 * dt)
    obs_used = obs + np.array([dx, 0, 0])
    pp = phi(t)
    obs_used = obs - np.array([dx, 0, 0])
    pm = phi(t)
    obs_used = obs.copy()
    return -rho0 * (pt + U0 * (pp - pm) / (2 * dx))



In [ ]:

t = 1000.0 + np.arange(2000)
# t = np.arange(3000)
pFVec = []
for case in range(0, len(fileVec)):
    d = np.genfromtxt(fileVec[case], skip_header=2)
    step, pFWH = d[:, 0].astype(int), d[:, 2]
    sIdx = {int(s): i for i, s in enumerate(step)}
    pF = np.array([pFWH[sIdx[int(tt)]] for tt in t])
    pFVec.append(pF)

# two full rotation periods after transients have passed
pA = np.array([p_prime(tt) for tt in t])


In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(10, 4), facecolor='w', edgecolor='w')
ax.plot(t, pA, c='k', lw=1.5, label='Analytic solution')
for case in range(0, len(fileVec)):
    ax.plot(t, pFVec[case], c=colorVec[case], label=labelVec[case], lw=2, linestyle='--')

ax.legend(loc='upper right', frameon=False, fontsize=13)
ax.set_xlim(1000, 3000)
ax.set_ylim(-5E-7, 5E-7)
ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$p\prime$")

In [ ]:

# print(f"rms:  FWH = {pF.std():.4e},  exact = {pA.std():.4e},  "
#       f"ratio = {pF.std() / pA.std():.4f}")
# print(f"{'l':>3} {'FWH amp':>11} {'exact amp':>11} {'ratio':>7} {'dphi[deg]':>9}")
# for l in range(8, 36):
#     e = np.exp(-1j * 2 * np.pi * l * (t - 1000.0) / 1000.0)
#     cF, cA = (pF @ e) * 2 / len(t), (pA @ e) * 2 / len(t)
#     if abs(cA) < 1e-10:
#         continue
#     print(f"{l:3d} {abs(cF):11.3e} {abs(cA):11.3e} "
#           f"{abs(cF) / abs(cA):7.3f} {np.degrees(np.angle(cF / cA)):9.2f}")